# 04 — Small MLP (PyTorch, CPU), validation only

**Design**
* Input: 2,000 HVGs, z-scored. HVG selection and scaling are fit on training cells only (same pipeline
  and split as Phase 3).
* Output: all 134 proteins. Targets are standardized per protein with the training mean/SD, so each
  protein weighs equally in the MSE loss. Predictions are mapped back to the CLR scale.
* Architecture: Linear -> ReLU -> Dropout blocks, AdamW, batch 512, lr 1e-3, at most 200 epochs.
  **Early stopping on validation loss** (patience 15); the best-epoch weights are kept.
* Search (seed 0): hidden layers {[256], [512, 256], [1024, 512]} x dropout {0.1, 0.3} x weight decay
  {1e-4, 1e-2}. The best configuration is chosen by the same metric as in Phase 3: mean validation
  Pearson over all 134 proteins. If the best value is on a grid edge, the grid is extended (validation
  only) and the extension is logged. Exception: the first hidden layer is capped at 2,048 units for CPU
  time. If the cap is hit, this is logged as well.
* The best configuration is trained with **3 seeds** (0, 1, 2), and results are reported as mean ± SD.
* Checkpoints go to `data/processed/phase4_checkpoints/`. A finished run is re-used if the notebook is
  run again.
* **Test cells are not used.** Validation decides both early stopping and the configuration, and the MLP
  search space is larger than ridge's. The MLP's validation numbers are therefore likely to be somewhat
  more optimistic than ridge's.
* Metrics as in Phase 3, plus within cell type x sample (notebook 03b).

In [ ]:
import os, sys, time, json, hashlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from citeseq.paths import RESULTS_DIR, FIGURES_DIR, PROCESSED_DIR
from citeseq.data_prep import load_train_val
from citeseq.baselines import HVGPCAFeatures, RidgePath
from citeseq.correlation import pearson_columns
from citeseq.metrics import evaluate
from citeseq.mlp import TargetScaler, train_mlp, predict
from citeseq import plotting as P

torch.set_num_threads(8)
SUBSET = int(os.environ.get("CITESEQ_SUBSET", 0)) or None
MAX_EPOCHS = int(os.environ.get("CITESEQ_MAX_EPOCHS", 200))
PATIENCE = 15
HIDDEN = [[256], [512, 256], [1024, 512]]
DROPOUT = [0.1, 0.3]
WEIGHT_DECAY = [1e-4, 1e-2]
SEEDS = [0, 1, 2]
MIN_CELLS = 50
MAX_EXPANSIONS = 3
MAX_WIDTH = 2048          # compute cap for the first hidden layer (CPU time)
cfg3 = json.loads((RESULTS_DIR / "phase3_frozen_configs.json").read_text())
CKPT = PROCESSED_DIR / ("phase4_checkpoints" if SUBSET is None else "test_phase4_checkpoints")
OUT_FIG = FIGURES_DIR if SUBSET is None else PROCESSED_DIR / "test_figures"
OUT_FIG.mkdir(parents=True, exist_ok=True)
P.set_style()
pd.set_option("display.float_format", "{:.3f}".format)
print("subset:", SUBSET, "| max epochs:", MAX_EPOCHS, "| torch", torch.__version__, "threads", torch.get_num_threads())

## 1. Inputs (train-only pipeline, same split as Phase 3)

In [ ]:
t0 = time.time()
d = load_train_val(RESULTS_DIR / "splits/random_split.csv.gz", subset=SUBSET)
proteins, pmap = d["proteins"], d["pmap"]
primary = pmap.index[pmap["analysis_set"] == "primary"].tolist()
feats = HVGPCAFeatures(n_top_genes=cfg3["features"]["hvg"], max_pcs=None, seed=cfg3["seed"]).fit(
    d["X_tr"], d["obs_tr"], d["gene_names"])
H_tr, H_va = feats.transform_hvg(d["X_tr"]), feats.transform_hvg(d["X_va"])
Y_tr, Y_va = d["Y_tr"], d["Y_va"]
ys = TargetScaler().fit(Y_tr)
Z_tr, Z_va = ys.transform(Y_tr), ys.transform(Y_va)
ct_va = d["obs_va"]["cell_type"].astype(str).to_numpy()
ts_va = (d["obs_va"]["cell_type"].astype(str) + " | " + d["obs_va"]["batch"].astype(str)).to_numpy()
print(f"inputs in {time.time() - t0:.0f}s | train {H_tr.shape} | val {H_va.shape} | test cells not loaded: {d['n_test']}")

# Ridge reference: the frozen Phase 3 model 3a (full-rank PCA)
n3a, a3a = cfg3["3a_ridge_pca"]["n_pcs"], cfg3["3a_ridge_pca"]["alpha"]
P_tr, P_va = feats.transform_pca(d["X_tr"], n3a), feats.transform_pca(d["X_va"], n3a)
pred_ridge = RidgePath().fit(P_tr, Y_tr, [a3a]).predict(P_va, a3a)
del P_tr, P_va
def sel_metric(p):
    return float(np.nanmean(pearson_columns(p, Y_va)))
print("ridge 3a selection metric:", round(sel_metric(pred_ridge), 4))

## 2. Search (seed 0)

In [ ]:
def ckpt_path(hidden, dropout, wd, seed):
    key = json.dumps([hidden, dropout, wd, seed, MAX_EPOCHS, PATIENCE])
    return CKPT / f"mlp_{hashlib.md5(key.encode()).hexdigest()[:10]}.pt"

runs = {}
def run(hidden, dropout, wd, seed):
    k = (tuple(hidden), dropout, wd, seed)
    if k in runs:
        return runs[k]
    t = time.time()
    out = train_mlp(H_tr, Z_tr, H_va, Z_va, hidden=hidden, dropout=dropout, weight_decay=wd, seed=seed,
                    max_epochs=MAX_EPOCHS, patience=PATIENCE, checkpoint=ckpt_path(hidden, dropout, wd, seed))
    p = predict(out["model"], H_va, ys)
    runs[k] = dict(hidden=str(hidden), dropout=dropout, weight_decay=wd, seed=seed, pred=p,
                   val_mean_pearson=sel_metric(p), best_epoch=out["best_epoch"],
                   epochs_run=len(out["history"]["val_loss"]), history=out["history"],
                   minutes=(time.time() - t) / 60, loaded=out["loaded"])
    r = runs[k]
    print(f"{str(hidden):<12} drop {dropout:<4} wd {wd:<7g} seed {seed}: val mean r {r['val_mean_pearson']:.4f}, "
          f"best epoch {r['best_epoch']}, {r['epochs_run']} epochs, {r['minutes']:.1f} min{' (loaded)' if r['loaded'] else ''}")
    return r

t0 = time.time()
hidden_grid, drop_grid, wd_grid = [list(h) for h in HIDDEN], list(DROPOUT), list(WEIGHT_DECAY)
for h in hidden_grid:
    for dr in drop_grid:
        for wd in wd_grid:
            run(h, dr, wd, 0)
expansion_log = []
def search_table():
    return pd.DataFrame([{k: v for k, v in r.items() if k not in ("pred", "history")} for r in runs.values() if r["seed"] == 0])
for _ in range(MAX_EXPANSIONS):
    st = search_table()
    b = st.loc[st["val_mean_pearson"].idxmax()]
    hb, drb, wdb = eval(b["hidden"]), float(b["dropout"]), float(b["weight_decay"])
    new = []
    if hb == hidden_grid[-1]:
        h2 = [2 * x for x in hidden_grid[-1]]
        if h2[0] <= MAX_WIDTH:
            hidden_grid.append(h2); new.append(("hidden", h2))
        elif not any("compute cap" in e for e in expansion_log):
            expansion_log.append(f"best hidden {hb} is the largest allowed; not extended to {h2} (compute cap {MAX_WIDTH})")
    elif hb == hidden_grid[0]:
        h2 = [max(32, x // 2) for x in hidden_grid[0]]; hidden_grid.insert(0, h2); new.append(("hidden", h2))
    if drb == max(drop_grid) and drb < 0.6:
        drop_grid.append(round(drb + 0.2, 2)); new.append(("dropout", drop_grid[-1]))
    elif drb == min(drop_grid) and drb > 0:
        drop_grid.insert(0, 0.0); new.append(("dropout", 0.0))
    if wdb == max(wd_grid):
        wd_grid.append(wdb * 10); new.append(("weight_decay", wd_grid[-1]))
    elif wdb == min(wd_grid):
        wd_grid.insert(0, wdb / 10); new.append(("weight_decay", wd_grid[0]))
    if not new:
        break
    expansion_log.append(f"best {hb}, dropout {drb}, wd {wdb:g} on a grid edge -> added {new}")
    # evaluate the new values around the current best (one dimension at a time)
    for kind, v in new:
        if kind == "hidden":
            run(v, drb, wdb, 0)
        elif kind == "dropout":
            run(hb, v, wdb, 0)
        else:
            run(hb, drb, v, 0)
print(f"search: {(time.time() - t0) / 60:.1f} min")
print("grid extensions (validation only):", expansion_log or "none")
search = search_table().sort_values("val_mean_pearson", ascending=False)
search

## 3. Best configuration with 3 seeds

In [ ]:
best = search.iloc[0]
best_cfg = dict(hidden=eval(best["hidden"]), dropout=float(best["dropout"]), weight_decay=float(best["weight_decay"]))
print("best configuration:", best_cfg)
final = [run(best_cfg["hidden"], best_cfg["dropout"], best_cfg["weight_decay"], s) for s in SEEDS]

In [ ]:
def summarize(p):
    e_t = evaluate(Y_va, p, ct_va, proteins, MIN_CELLS)
    e_ts = evaluate(Y_va, p, ts_va, proteins, MIN_CELLS)
    e_t["within_type_sample_w"] = e_ts["within_w"]
    return e_t

per_seed = {s: summarize(r["pred"]) for s, r in zip(SEEDS, final)}
ridge_eval = summarize(pred_ridge)
metrics = ["pearson_all", "within_w", "within_unw", "within_type_sample_w", "rmse"]
seed_medians = pd.DataFrame({s: e.loc[primary, metrics].median() for s, e in per_seed.items()}).T
tab = pd.DataFrame({
    "MLP mean": seed_medians.mean(), "MLP SD (3 seeds)": seed_medians.std(ddof=1),
    "ridge 3a (Phase 3)": ridge_eval.loc[primary, metrics].median(),
})
tab["MLP - ridge"] = tab["MLP mean"] - tab["ridge 3a (Phase 3)"]
print(f"PRIMARY set ({len(primary)} proteins), validation; median across proteins per seed, then mean ± SD over seeds")
tab

In [ ]:
# Per protein: MLP (mean over seeds) vs ridge
mlp_pp = sum(per_seed[s][metrics] for s in SEEDS) / len(SEEDS)
for m in ["pearson_all", "within_w", "within_type_sample_w"]:
    dlt = (mlp_pp.loc[primary, m] - ridge_eval.loc[primary, m])
    print(f"{m:<22} MLP - ridge: median {dlt.median():+.3f}; MLP better for {(dlt > 0).sum()} of {dlt.notna().sum()}")
for name, sel in [("secondary", "secondary"), ("unavailable", "unavailable")]:
    prots = pmap.index[pmap["analysis_set"] == sel].tolist()
    print(f"{name:<11} set: MLP pearson_all {mlp_pp.loc[prots, 'pearson_all'].median():.3f} vs ridge {ridge_eval.loc[prots, 'pearson_all'].median():.3f}; "
          f"within {mlp_pp.loc[prots, 'within_w'].median():.3f} vs {ridge_eval.loc[prots, 'within_w'].median():.3f}")

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, len(final), figsize=(4 * len(final), 3.2), sharey=True)
for ax, r in zip(np.atleast_1d(axes), final):
    h = r["history"]
    ep = np.arange(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], color=P.BLUE, lw=2, label="train")
    ax.plot(ep, h["val_loss"], color=P.ORANGE, lw=2, label="validation")
    ax.axvline(r["best_epoch"] + 1, color=P.MUTED, ls="--", lw=1)
    ax.text(r["best_epoch"] + 1, ax.get_ylim()[1], " best", va="top", fontsize=8, color=P.INK_2)
    ax.set_title(f"seed {r['seed']}"); ax.set_xlabel("epoch")
np.atleast_1d(axes)[0].set_ylabel("MSE (standardized targets)")
np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
fig.suptitle(f"MLP {best_cfg['hidden']}, dropout {best_cfg['dropout']}, weight decay {best_cfg['weight_decay']:g}", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_FIG / "phase4_training_curves.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m, title in zip(axes, ["pearson_all", "within_w", "within_type_sample_w"],
                        ["all validation cells", "within cell type", "within cell type x sample"]):
    x, y = ridge_eval.loc[primary, m], mlp_pp.loc[primary, m]
    lo = min(np.nanmin(x), np.nanmin(y), 0) - 0.05
    ax.plot([lo, 1], [lo, 1], color=P.MUTED, ls="--", lw=1)
    ax.scatter(x, y, s=18, c=P.BLUE, edgecolors=P.SURFACE, linewidths=0.6)
    ax.set_xlabel("ridge 3a, Pearson r"); ax.set_ylabel("MLP (mean of 3 seeds), Pearson r"); ax.set_title(title)
fig.suptitle("Per protein (primary set), validation", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_FIG / "phase4_mlp_vs_ridge.png")

In [ ]:
frozen = {
    "split_file": "results/splits/random_split.csv.gz", "input": "2000 HVGs, z-scored (train-only)",
    "targets": "per-protein standardized CLR (train mean/SD)",
    "selection_metric": "mean validation Pearson over all 134 proteins (best-epoch weights)",
    "early_stopping": {"monitor": "validation MSE", "patience": PATIENCE, "max_epochs": MAX_EPOCHS},
    "grid": {"hidden": hidden_grid, "dropout": drop_grid, "weight_decay": wd_grid},
    "grid_extensions_based_on_validation": expansion_log,
    "hidden_width_cap": MAX_WIDTH,
    "best": {**best_cfg, "lr": 1e-3, "batch_size": 512, "optimizer": "AdamW"},
    "seeds": SEEDS, "best_epochs": [r["best_epoch"] for r in final],
}
if SUBSET is None:
    (RESULTS_DIR / "phase4_frozen_config.json").write_text(json.dumps(frozen, indent=2))
    search.to_csv(RESULTS_DIR / "phase4_search.csv", index=False)
    pd.concat({f"seed{s}": e.assign(analysis_set=pmap["analysis_set"]) for s, e in per_seed.items()},
              names=["seed"]).reset_index().to_csv(RESULTS_DIR / "phase4_val_per_protein.csv", index=False)
    np.savez_compressed(PROCESSED_DIR / "phase4_val_predictions.npz", proteins=np.array(proteins),
                        **{f"mlp_seed{r['seed']}": r["pred"].astype(np.float32) for r in final})
frozen

In [ ]:
import resource, platform
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory: {(peak / 1e9 if platform.system() == 'Darwin' else peak * 1024 / 1e9):.2f} GB")